# 02. Bundle synthesis with approximation

This notebook implements the **bundle-of-trajectories** stage:
1. Build a bundle from top-scoring trajectories.
2. Train a quadratic control approximation per time step.
3. Generate trajectories with the approximated feedback law.
4. Evaluate score quality and visualize approximation behavior.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base = Path.cwd().resolve()
if (base / 'src').exists():
    ROOT = base
elif (base / 'research' / 'src').exists():
    ROOT = base / 'research'
elif (base.parent / 'src').exists():
    ROOT = base.parent
else:
    raise RuntimeError('Cannot locate research/src directory from current working directory')

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))
from dynamics import CostConfig, build_trajectory_bundle, bolza_cost, clamp_controls, load_training_samples, rollout

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)


In [ ]:
DATA_DIR = ROOT / 'src' / 'data'
samples = load_training_samples(DATA_DIR)
bundle = build_trajectory_bundle(samples)
score_table = samples[['trajectory_id', 'score']].drop_duplicates().sort_values('score').reset_index(drop=True)

print('All trajectories with score:', len(score_table))
print(score_table.head(10))


In [ ]:
# Build the top-K bundle (best trajectories) for approximation.
TOP_K = 160
selected_ids = score_table.head(TOP_K)['trajectory_id'].astype(int).tolist()
selected = {tid: bundle[tid] for tid in selected_ids}

cfg = CostConfig()
N_STEPS = cfg.num_intervals

print('Selected trajectories:', len(selected))
print('Average score in selected bundle:', np.mean([selected[tid]['score'] for tid in selected]))
print('Worst score in selected bundle:', np.max([selected[tid]['score'] for tid in selected]))


In [ ]:
# Split by trajectory ID to avoid leakage across steps.
ids = np.array(selected_ids)
np.random.shuffle(ids)
train_cut = int(0.8 * len(ids))
train_ids = ids[:train_cut]
test_ids = ids[train_cut:]

print('Train trajectories:', len(train_ids))
print('Test trajectories:', len(test_ids))


In [ ]:
class QuadraticBundleController:
    def __init__(self, m_features=6, ridge_lambda=1e-3):
        self.m = int(m_features)
        self.ridge_lambda = float(ridge_lambda)
        self.models = {}

    def _basis(self, X_batch):
        x = X_batch[:, :self.m]
        quad = []
        for i in range(self.m):
            for j in range(i, self.m):
                col = x[:, i] * x[:, j]
                if i == j:
                    col = 0.5 * col
                quad.append(col)
        return np.column_stack(quad + [x, np.ones(len(x))])

    def fit(self, bundle_dict, trajectory_ids, n_steps):
        self.models = {}
        for step in range(1, n_steps + 1):
            X_list, U_list = [], []
            for tid in trajectory_ids:
                row = bundle_dict[int(tid)]
                if step <= len(row['X']) and step <= len(row['U']):
                    X_list.append(row['X'][step - 1])
                    U_list.append(row['U'][step - 1])
            if not X_list:
                continue

            Xs = np.vstack(X_list)
            Us = np.vstack(U_list)
            G = self._basis(Xs)

            reg = self.ridge_lambda * np.eye(G.shape[1])
            coeff = np.linalg.solve(G.T @ G + reg, G.T @ Us)
            self.models[step] = coeff

    def predict(self, x_state, step):
        if step not in self.models:
            return np.zeros(4)

        x = np.asarray(x_state[:self.m], dtype=float)
        basis = []
        for i in range(self.m):
            for j in range(i, self.m):
                basis.append(0.5 * x[i] * x[i] if i == j else x[i] * x[j])
        basis.extend(x.tolist())
        basis.append(1.0)
        u = np.asarray(basis) @ self.models[step]
        return clamp_controls(u)


controller = QuadraticBundleController(m_features=6, ridge_lambda=2e-3)
controller.fit(selected, train_ids, N_STEPS)
print('Trained step models:', len(controller.models), 'out of', N_STEPS)


In [ ]:
# Pointwise approximation metrics on held-out trajectories.
errors = []
for tid in test_ids:
    row = selected[int(tid)]
    X, U = row['X'], row['U']
    for step in range(1, min(len(X), len(U), N_STEPS) + 1):
        u_hat = controller.predict(X[step - 1], step)
        errors.append((u_hat - U[step - 1]) ** 2)

errors = np.asarray(errors)
rmse_per_control = np.sqrt(errors.mean(axis=0))
print('Control RMSE [phi, theta, psi, thrust]:', np.round(rmse_per_control, 5))
print('Global RMSE:', float(np.sqrt(errors.mean())))


In [ ]:
# Closed-loop synthesis from multiple initial states.
def synthesize_with_controller(initial_state, controller, cfg):
    x = np.asarray(initial_state, dtype=float)
    controls = []
    for step in range(1, cfg.num_intervals + 1):
        u = controller.predict(x, step)
        controls.append(u)
        x = rollout(x, np.asarray([u]), cfg.dt)[-1]
    controls = np.asarray(controls)
    states = rollout(initial_state, controls, cfg.dt)
    score = bolza_cost(initial_state, controls, cfg)
    return states, controls, score

# Evaluate on held-out initial conditions.
results = []
for tid in test_ids[: min(60, len(test_ids))]:
    row = selected[int(tid)]
    x0 = row['X'][0]
    states, controls, score = synthesize_with_controller(x0, controller, cfg)
    results.append({'trajectory_id': int(tid), 'pred_score': score, 'true_score': row['score'], 'states': states, 'controls': controls})

res_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('states', 'controls')} for r in results])
print(res_df[['pred_score', 'true_score']].describe())
print('Mean score gap (pred - true):', float((res_df['pred_score'] - res_df['true_score']).mean()))


In [ ]:
# Visualize bundle and synthesized trajectories in x-z plane.
fig, ax = plt.subplots(figsize=(8, 7))

for tid in train_ids[:40]:
    X = selected[int(tid)]['X']
    ax.plot(X[:, 0], X[:, 2], color='#9AA0A6', alpha=0.22, linewidth=1)

for r in results[:10]:
    S = r['states']
    ax.plot(S[:, 0], S[:, 2], color='#E63946', alpha=0.9, linewidth=1.8)

for cyl in cfg.cylinders:
    c = plt.Circle((cyl.x, cyl.z), cyl.radius, color='#F4A261', alpha=0.25)
    ax.add_patch(c)
for wnd in cfg.windows:
    w = plt.Circle((wnd.x, wnd.z), wnd.radius, color='#2A9D8F', alpha=0.35)
    ax.add_patch(w)

ax.scatter([cfg.terminal_state[0]], [cfg.terminal_state[2]], marker='*', s=180, color='black', label='terminal')
ax.set_title('Top-K bundle (gray) and synthesized trajectories (red)')
ax.set_xlabel('x1')
ax.set_ylabel('x3')
ax.axis('equal')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


In [ ]:
# Predicted-vs-true score comparison.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].scatter(res_df['true_score'], res_df['pred_score'], alpha=0.8, color='#1D3557')
line_min = min(res_df['true_score'].min(), res_df['pred_score'].min())
line_max = max(res_df['true_score'].max(), res_df['pred_score'].max())
ax[0].plot([line_min, line_max], [line_min, line_max], '--', color='crimson')
ax[0].set_title('Score agreement on held-out initials')
ax[0].set_xlabel('true score (optimized)')
ax[0].set_ylabel('predicted closed-loop score')

ax[1].boxplot([res_df['true_score'], res_df['pred_score']], labels=['true', 'pred'])
ax[1].set_title('Score distributions')
ax[1].set_ylabel('score')

plt.tight_layout()
plt.show()


In [ ]:
# Control-level fit diagnostics: one control example (thrust).
all_true = []
all_pred = []
for tid in test_ids:
    row = selected[int(tid)]
    for step in range(1, min(len(row['X']), len(row['U']), N_STEPS) + 1):
        all_true.append(row['U'][step - 1, 3])
        all_pred.append(controller.predict(row['X'][step - 1], step)[3])

all_true = np.asarray(all_true)
all_pred = np.asarray(all_pred)

plt.figure(figsize=(6, 5))
plt.scatter(all_true, all_pred, s=10, alpha=0.55, color='#457B9D')
mn, mx = float(min(all_true.min(), all_pred.min())), float(max(all_true.max(), all_pred.max()))
plt.plot([mn, mx], [mn, mx], '--', color='crimson')
plt.xlabel('true thrust')
plt.ylabel('predicted thrust')
plt.title('Approximation quality for thrust channel')
plt.tight_layout()
plt.show()


### Result summary

- Built a trajectory bundle from top-scoring runs.
- Trained a quadratic per-step control approximation on trajectory states.
- Synthesized new trajectories in closed loop and compared their Bolza scores to true optimized trajectories.
- Produced geometry and regression diagnostics to support analysis in paper form.
